# Init

In [17]:
# init
from importlib import reload
import os
from pathlib import Path
import pandas as pd
import sys

import toolsGeneral.main as tgm
import toolsGeneral.logger as tgl
import toolsOSM.overpass as too
import toolsPandas.helpers as tph
import toolsSync.main as tsm

def pckgs_reload():
    reload(tgm)
    reload(too)
    reload(tph)
    reload(tgl)
    reload(tsm)

ROOT = Path.cwd().parents[0]
DATA_DIR = ROOT / "data"
TESTS_DIR = DATA_DIR / 'tests results'
CLEANED_DIR = DATA_DIR / 'cleaned'
TEST_BASIC_DIR = TESTS_DIR / 'osm basic test'
TEST_DUPLICATES_DIR = TESTS_DIR / 'osm duplicates test'
TEST_FIRST_LEVEL_DIR = TESTS_DIR / 'osm first level test'

logger = tgl.initiate_logger('logger', TEST_BASIC_DIR / 'basic_test.log')

In [ ]:
to_review_triplets = [('72639','60189','60189'), ('72639','60199','60199'),('6543282', '6543265', '192756'),
 ('6543282', '6543273', '192756'), ('59190', '59195', '59065'), ('59190', '59752', '59065'), ('8309397', '391455', '148838'), ('8309397', '1116270', '148838')]

In [ ]:
tgm.dump(TEST_DUPLICATES_DIR / "to_review_triplets.pkl", to_review_triplets)

In [7]:
df_by_cntr = tgm.load_dirs(CLEANED_DIR)
logger.info(f"Cleaned data for countries: {len(df_by_cntr)}")

[2025-12-25 15:53:21] [INFO] : Cleaned data for countries: 83


In [8]:
id_triplets = pd.concat(df_by_cntr.values(), ignore_index=True)[['id', 'tags.parent_id', 'tags.country_id']].fillna('missing').apply(tuple,axis=1).to_list()
logger.info(f"All dataframes id triplets: {len(id_triplets)}")

[2025-12-25 15:53:27] [INFO] : All dataframes id triplets: 155843


# basic test

### * review

In [9]:
test_res_by_cntr = tgm.load_dirs(TEST_BASIC_DIR)
logger.info(f'Test results found: {len(test_res_by_cntr)}')

[2025-12-25 15:53:30] [INFO] : Test results found: 83


In [3]:
temp = pd.DataFrame(test_res_by_cntr)
temp = temp.T.map(len)
temp['test_tags_total'] = [len(elems) for c,elems in df_by_cntr.items() if c in test_res_by_cntr.keys()]
temp.peek()

,missing_name,test_tags_leak,test_tags_in_country,test_tags_NA_result,test_tags_total
Afghanistan,0,0,24,11,35
Albania,0,0,390,0,390
Algeria,0,0,1,2147,2148
Andorra,0,0,1,0,1
Angola,0,0,1,63,64
Anguilla,0,0,1,0,1
AntiguaAndBarbuda,0,0,1,11,12
Argentina,1,0,885,326,1211
Armenia,0,0,27,0,27
Australia,0,0,1,615,616


### * select missing names and leaks from other countries

In [5]:
missing_names = set()
leaks = set()
for k,v in test_res_by_cntr.items():
    missing_names.update(v['missing_name'])
for k,v in test_res_by_cntr.items():
    leaks.update(v['test_tags_leak'])

logger.info(f"Missing names relations: {len(missing_names)}")
logger.info(f"Leaks relations: {len(leaks)}")
relations_from_test_to_delete = leaks | missing_names
logger.info(f"To delete relations (parents): {len(relations_from_test_to_delete)}")

[2025-12-25 15:50:46] [INFO] : Missing names relations: 23
[2025-12-25 15:50:46] [INFO] : Leaks relations: 20
[2025-12-25 15:50:46] [INFO] : To delete relations (parents): 43


### * from the relations to delete, select the childs in the country that has them as parent

In [12]:
parents_to_delete = relations_from_test_to_delete
relations_childs_to_delete = set()
while len(parents_to_delete) > 0:
    parents_id_and_countryid = {(ele[0],ele[2]) for ele in parents_to_delete}
    childs_to_delete = {ele for ele in id_triplets if (ele[1], ele[2]) in parents_id_and_countryid}
    relations_childs_to_delete.update(childs_to_delete)
    parents_to_delete = childs_to_delete
logger.info(f"Childs to delete: {len(relations_childs_to_delete)}")

[2025-12-25 15:54:06] [INFO] : Childs to delete: 4161


### * dump basic_test_to_delete.json

In [13]:
basic_test_to_delete = relations_from_test_to_delete | relations_childs_to_delete
logger.info(f"Basic test to delete relations: {len(basic_test_to_delete)}")

[2025-12-25 15:56:40] [INFO] : Basic test to delete relations: 4199


In [14]:
tgm.dump(TEST_BASIC_DIR / "basic_test_to_delete.pkl", basic_test_to_delete)

# first level test

### * review

In [19]:
first_level_test_res = tgm.load_dirs(TEST_FIRST_LEVEL_DIR)
logger.info(f'First level test results found: {len(first_level_test_res)}')

[2025-12-25 16:06:28] [INFO] : First level test results found: 68


In [20]:
first_level_test_res['Afghanistan']

{('1674535',
  '303427',
  '303427'): {'admin_centre': {'status': 'ok',
   'result': True,
   'status_type': None,
   'node': ['admin_centre', 312586412]}, 'label': {'status': 'ok',
   'result': True,
   'status_type': None,
   'node': ['label', 312470052]}, 'place': {'status': 'ok',
   'result': True,
   'status_type': None,
   'node': ['place', 312470052]}, 'geom_node': {'status': 'ok',
   'result': True,
   'status_type': None,
   'node': ['geom_node', 271697700]}, 'centroid': {'status': 'ok',
   'result': True,
   'status_type': None,
   'node': ['centroid', [36.9672214, 72.435006]]}},
 ('1674541',
  '303427',
  '303427'): {'admin_centre': {'status': 'ok',
   'result': True,
   'status_type': None,
   'node': ['admin_centre', 318091652]}, 'label': {'status': 'ok',
   'result': True,
   'status_type': None,
   'node': ['label', 312470050]}, 'place': {'status': 'ok',
   'result': True,
   'status_type': None,
   'node': ['place', 312470050]}, 'geom_node': {'status': 'ok',
   'result'

In [31]:
cntr_meta = tgm.load(r"C:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\osmMetaCountrDict.json")

In [32]:
cntr_meta

{'Andorra': {'id': '9407', 'addLvlsNum': ['4', '6', '8']},
 'Slovakia': {'id': '14296', 'addLvlsNum': ['4', '6', '8']},
 'Austria': {'id': '16239', 'addLvlsNum': ['4', '6', '8']},
 'Germany-Austria': {'id': '17511', 'addLvlsNum': ['4', '6', '8']},
 'Hungary': {'id': '21335', 'addLvlsNum': ['4', '6', '8']},
 'Georgia': {'id': '28699', 'addLvlsNum': ['4', '6', '8']},
 'VaticanCity': {'id': '36989', 'addLvlsNum': ['4', '6', '8']},
 'Poland': {'id': '49715', 'addLvlsNum': ['4', '6', '8']},
 'Cambodia': {'id': '49898', 'addLvlsNum': ['4', '6', '8']},
 'Laos': {'id': '49903', 'addLvlsNum': ['4', '6', '8']},
 'Vietnam': {'id': '49915', 'addLvlsNum': ['4', '6', '8']},
 'Denmark': {'id': '50046', 'addLvlsNum': ['4', '6', '8']},
 'Myanmar': {'id': '50371', 'addLvlsNum': ['4', '6', '8']},
 'Germany--Switzerland': {'id': '51239', 'addLvlsNum': ['4', '6', '8']},
 'Germany': {'id': '51477', 'addLvlsNum': ['4', '6', '8']},
 'Czechia': {'id': '51684', 'addLvlsNum': ['4', '6', '8']},
 'Switzerland': {'

In [28]:
first_level_test_res_by_elem = {k:v for list in first_level_test_res.values() for k,v in list.items()}
print(len(first_level_test_res_by_elem))

1256


In [58]:
less = set()
for elem, val in first_level_test_res_by_elem.items():
    if len(val) < 4: less.add(elem)
print(len(less))
{id_to_cntr[id[2]] for id in less}

67


{'Armenia',
 'Bahrain',
 'Belize',
 'Benin',
 'Bolivia',
 'BosniaAndHerzegovina',
 'Botswana',
 'BritishVirginIslands'}

In [48]:
cntr_meta = tgm.load(r"C:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\osmMetaCountrDict.json")
id_to_cntr = {val['id']:cntr for cntr, val in cntr_meta.items()}

In [53]:
id_to_cntr['1889339']

'Botswana'

In [36]:
temp = too.is_child_inside_parent('364077', '364066')

[2025-12-25 16:30:13] [INFO] :    > Getting admin_centre:
[2025-12-25 16:30:14] [INFO] :     * Found node (admin_centre)
[2025-12-25 16:30:14] [INFO] :    > Testing admin_centre node (id: 130037434)
[2025-12-25 16:30:19] [INFO] :    * Attempt 1 failed: http_error: 504
[2025-12-25 16:30:21] [INFO] :     * Finished testing (admin_centre): True
[2025-12-25 16:30:21] [INFO] :    > Getting label:
[2025-12-25 16:30:22] [INFO] :     * Missing node (label)
[2025-12-25 16:30:22] [INFO] :    > Getting place:
[2025-12-25 16:30:27] [INFO] :    * Attempt 1 failed: http_error: 429
[2025-12-25 16:30:29] [INFO] :     * Found node (place)
[2025-12-25 16:30:29] [INFO] :    > Testing place node (id: 130037434)
[2025-12-25 16:30:33] [INFO] :     * Finished testing (place): True
[2025-12-25 16:30:33] [INFO] :    > Getting geom_node:
[2025-12-25 16:30:38] [INFO] :    * Attempt 1 failed: http_error: 504
[2025-12-25 16:30:44] [INFO] :    * Attempt 2 failed: http_error: 504
[2025-12-25 16:30:51] [INFO] :    * 

In [29]:
result_status = [[[k,v['status']] for k,v in ele.items()] for ele in first_level_test_res_by_elem.values()]
pd.Series(list(result_status)).value_counts()

[[admin_centre, ok], [label, ok], [place, ok], [geom_node, ok], [centroid, ok]]                   721
[[admin_centre, ok], [label, missing], [place, ok], [geom_node, ok], [centroid, ok]]              459
[[admin_centre, ok], [label, missing]]                                                             66
[[admin_centre, missing], [label, missing], [place, missing], [geom_node, ok], [centroid, ok]]      5
[[admin_centre, ok], [label, ok], [place, error], [geom_node, ok], [centroid, ok]]                  2
[[admin_centre, error], [label, missing]]                                                           1
[[admin_centre, missing], [label, ok], [place, ok], [geom_node, ok], [centroid, ok]]                1
[[admin_centre, ok], [label, ok], [place, ok], [geom_node, error], [centroid, ok]]                  1
Name: count, dtype: int64

In [30]:
pd.Series([ele[1] for lis in result_status for ele in lis]).value_counts()

ok         5533
missing     542
error         4
Name: count, dtype: int64

In [27]:
import json
print(json.dumps(first_level_test_res_by_tuple[('364077', '364066', '364066')], indent=4))

{
    "admin_centre": {
        "status": "error",
        "result": true,
        "status_type": null,
        "node": [
            "admin_centre",
            208840062
        ]
    },
    "label": {
        "status": "missing",
        "result": null,
        "status_type": "missing_node",
        "node": [
            "label",
            null
        ]
    }
}


### * select false entities to delete

In [22]:
MIN_TESTS = 2
KEEP_THRESHOLD = 0.3
miss = 0
test_res_by_tuple_bool = {}
for country, res in first_level_test_res.items():
    for id, val in res.items():
        true_count = 0
        false_count = 0
        for type in ['admin_centre', 'label', 'place', 'geom_node', 'centroid']:
            try:
                if val[type]['status'] == 'ok':
                    # make a voting weight selection
                    if val[type]['result'] is True:
                        true_count += 1
                    else:
                        false_count += 1
            except:
                print(id)
        if (true_count + false_count) < MIN_TESTS:
            test_res_by_tuple_bool[id] = 'missing'
        else:
            true_ratio = true_count / ((false_count + true_count))
            # test_res_by_tuple_bool[id] = true_count >= false_count
            test_res_by_tuple_bool[id] = true_ratio > KEEP_THRESHOLD

logger.info(f"Data types: {tgm.tally([v for k,v in test_res_by_tuple_bool.items()])}")

[2025-12-25 16:12:35] [INFO] : Data types: Counter({True: 1176, 'missing': 67, False: 13})


('364077', '364066', '364066')
('364077', '364066', '364066')
('364077', '364066', '364066')
('364078', '364066', '364066')
('364078', '364066', '364066')
('364078', '364066', '364066')
('364079', '364066', '364066')
('364079', '364066', '364066')
('364079', '364066', '364066')
('364080', '364066', '364066')
('364080', '364066', '364066')
('364080', '364066', '364066')
('364081', '364066', '364066')
('364081', '364066', '364066')
('364081', '364066', '364066')
('364082', '364066', '364066')
('364082', '364066', '364066')
('364082', '364066', '364066')
('364083', '364066', '364066')
('364083', '364066', '364066')
('364083', '364066', '364066')
('364084', '364066', '364066')
('364084', '364066', '364066')
('364084', '364066', '364066')
('364086', '364066', '364066')
('364086', '364066', '364066')
('364086', '364066', '364066')
('364087', '364066', '364066')
('364087', '364066', '364066')
('364087', '364066', '364066')
('3917110', '364066', '364066')
('3917110', '364066', '364066')
('3917

In [ ]:
relations_from_test_to_delete = {k for k,v in test_res_by_tuple_bool.items() if v is False}
logger.info(f'Relations from test to delete: {len(relations_from_test_to_delete)}')

[INFO] : Relations from test to delete: 0


### * From the relations to delete, select the childs in the country that has them as parent

In [ ]:
id_triplets = pd.concat(countries_to_test_df.values(), ignore_index=True)[['id', 'tags.parent_id', 'tags.country_id']].fillna('missing').apply(tuple,axis=1).to_list()
logger.info(f"All dataframes id triplets: {len(id_triplets)}")

[INFO] : All dataframes id triplets: 27


In [ ]:
parents_to_delete = relations_from_test_to_delete
relations_childs_to_delete = set()
while len(parents_to_delete) > 0:
    parents_id_and_countryid = {(ele[0],ele[2]) for ele in parents_to_delete}
    childs_to_delete = {ele for ele in id_triplets if (ele[1], ele[2]) in parents_id_and_countryid}
    relations_childs_to_delete.update(childs_to_delete)
    parents_to_delete = childs_to_delete
logger.info(f"Childs to delete: {len(relations_childs_to_delete)}")

[INFO] : Childs to delete: 0


In [ ]:
logger.info(f"Old total of relations to delete from first level test: {len(first_level_test_to_delete)}")

[INFO] : Old total of relations to delete from first level test: 4932


In [ ]:
relations_to_delete = relations_from_test_to_delete | relations_childs_to_delete
logger.info(f"Current relations to delete: {len(relations_to_delete)}")

[INFO] : Current relations to delete: 0


In [ ]:
first_level_test_to_delete.update(relations_to_delete)
logger.info(f"New total of relations to delete from first level test: {len(first_level_test_to_delete)}")

[INFO] : New total of relations to delete from first level test: 4932


In [ ]:
tgm.dump(DATA_DIR  / "first_level_test_to_delete.json", first_level_test_to_delete)

# duplicates test

### * review

In [15]:
dups_test_res_by_cntr = tgm.load_dirs(TEST_DUPLICATES_DIR)
logger.info(f'Dups test results found: {len(dups_test_res_by_cntr)}')

NameError: name 'TEST_DUPLICATES_DIR' is not defined

In [ ]:
files = [f for f in (TEST_RESULTS_DIR / 'osm duplicates test').glob('*/*')]
print(len(files))
test_res_by_cntr = {}
for f in files:
    test_res_by_cntr[f.parent.name] = tgm.load(str(f))

print(len(test_res_by_cntr))

test_res_by_elem = {k:v for list in test_res_by_cntr.values() for k,v in list.items()}
print(len(test_res_by_elem))

30
30
1819


In [ ]:
result_status = [[[k,v['status']] for k,v in ele.items()] for ele in test_res_by_elem.values()]
pd.Series(list(result_status)).value_counts()

[[admin_centre, ok], [label, missing], [place, ok], [geom_node, ok], [centroid, ok]]                        1480
[[admin_centre, missing], [label, missing], [place, missing], [geom_node, ok], [centroid, ok]]               154
[[admin_centre, missing], [label, ok], [place, ok], [geom_node, ok], [centroid, ok]]                         151
[[admin_centre, ok], [label, ok], [place, ok], [geom_node, ok], [centroid, ok]]                               26
[[admin_centre, missing], [label, missing], [place, ok], [geom_node, ok], [centroid, ok]]                      6
[[admin_centre, missing], [label, missing], [place, missing], [geom_node, missing], [centroid, missing]]       2
Name: count, dtype: int64

In [ ]:
pd.Series([ele[1] for lis in result_status for ele in lis]).value_counts()

ok         6980
missing    2115
Name: count, dtype: int64

### * select false entities and childs to delete

In [ ]:
test_res_simplified = {}
for id, val in test_res_by_elem.items():
    true_count = 0
    false_count = 0
    for type in ['admin_centre', 'label', 'place', 'geom_node', 'centroid']:
        if val[type]['status'] == 'ok':
            # select the first one that gives True
            # test_res_simplified[id] = False
            # if val[type]['result'] is True:
            #     test_res_simplified[id] = val[type]['result']
            #     break

            # assign the first valid result
            # test_res_simplified[id] = val[type]['result']
            # break

            # make a voting weight selection
            if val[type]['result'] is True: true_count += 1
            else: false_count += 1
    
    test_res_simplified[id] = true_count >= false_count


pd.Series([v for k,v in test_res_simplified.items()]).value_counts()

True     1616
False     189
Name: count, dtype: int64

In [ ]:
dups_tuples_to_delete_parents = [k for k,v in test_res_simplified.items() if v is False]
print(len(dups_tuples_to_delete_parents))

189


In [ ]:
temp = [[ele[0],ele[2]] for ele in dups_tuples_to_delete_parents]
dups_tuples_to_delete_childs = [ele for ele in id_tuples if [ele[1],ele[2]] in temp]
print(len(dups_tuples_to_delete_childs))

# dups_tuples_to_delete = dups_tuples_to_delete_childs + dups_tuples_to_delete_parents
dups_tuples_to_delete = dups_tuples_to_delete_parents
print(len(dups_tuples_to_delete))

930
189


In [ ]:
tgm.dump(os.path.join(os.getcwd(), "../data/tests results/osm duplicates test", "dups_tuples_to_delete.pkl"), dups_tuples_to_delete)

### * add manual review deletion

In [ ]:
to_review_ids = tgm.load(os.path.join(os.getcwd(), "../data/tests results/osm duplicates test", "to_review_ids.pkl"))
print(len(to_review_ids))

6


In [ ]:
dups_tuples_to_delete_manual = [('6543282', '6543273', '192756'), ('59190', '59195', '59065')]

In [ ]:
tgm.dump(os.path.join(os.getcwd(), "../data/tests results/osm duplicates test", "dups_tuples_to_delete_manual.pkl"), dups_tuples_to_delete_manual)

In [ ]:
dups_tuples_to_delete = dups_tuples_to_delete + dups_tuples_to_delete_manual
print(len(dups_tuples_to_delete))

191


In [ ]:
tgm.dump(os.path.join(os.getcwd(), "../data/tests results/osm duplicates test", "dups_tuples_to_delete.pkl"), dups_tuples_to_delete)

In [ ]:
print(len(dups_id_tuples))
print(len(dups_tuples_to_delete))
temp = tgm.complement(dups_id_tuples, dups_tuples_to_delete)
print(len(temp))

1805
191
1614


# apply fixes

In [14]:
basic_to_delete = tgm.load(os.path.join(os.getcwd(), "../data/tests results/osm basic test", "basic_to_delete.pkl"))
print(len(basic_to_delete))

dups_tuples_to_delete = tgm.load(os.path.join(os.getcwd(), "../data/tests results/osm duplicates test", "dups_tuples_to_delete.pkl"))
print(len(dups_tuples_to_delete))

tuples_to_delete = tgm.load(os.path.join(os.getcwd(), "../data/tests results/osm first level center test", "tuples_to_delete.pkl"))
print(len(tuples_to_delete))

865
120
87


In [15]:
to_delete = list(set(basic_to_delete) | set(dups_tuples_to_delete) | set(tuples_to_delete))
print(len(to_delete))

1041


In [16]:
df_by_cntr_fixed = {}
for k,df in df_by_cntr.items():
    df_by_cntr_fixed[k] = df[~df[['id','tags.parent_id','tags.country_id']].apply(tuple, axis=1).isin(to_delete)]
print(len(df_by_cntr_fixed))

64


In [17]:
df_all_fixed =  pd.concat(df_by_cntr_fixed.values(), ignore_index=True)
print(len(df_all_fixed))

117697


In [18]:
print(len(df_all))
print(len(df_all)-len(to_delete))
print(len(df_all_fixed))

118738
117697
117697


# derivate data, duplicates test

### * select false entities and childs to delete

In [ ]:
test_res_simplified = {}
for id, val in test_res_by_elem.items():
    true_count = 0
    false_count = 0
    for type in ['admin_centre', 'label', 'place', 'geom_node', 'centroid']:
        if val[type]['status'] == 'ok':
            # select the first one that gives True
            # test_res_simplified[id] = False
            # if val[type]['result'] is True:
            #     test_res_simplified[id] = val[type]['result']
            #     break

            # assign the first valid result
            # test_res_simplified[id] = val[type]['result']
            # break

            # make a voting weight selection
            if val[type]['result'] is True: true_count += 1
            else: false_count += 1
    
    test_res_simplified[id] = true_count >= false_count


pd.Series([v for k,v in test_res_simplified.items()]).value_counts()

True     1616
False     189
Name: count, dtype: int64

In [ ]:
dups_tuples_to_delete_parents = [k for k,v in test_res_simplified.items() if v is False]
print(len(dups_tuples_to_delete_parents))

189


In [ ]:
temp = [[ele[0],ele[2]] for ele in dups_tuples_to_delete_parents]
dups_tuples_to_delete_childs = [ele for ele in id_tuples if [ele[1],ele[2]] in temp]
print(len(dups_tuples_to_delete_childs))

# dups_tuples_to_delete = dups_tuples_to_delete_childs + dups_tuples_to_delete_parents
dups_tuples_to_delete = dups_tuples_to_delete_parents
print(len(dups_tuples_to_delete))

930
189


In [ ]:
tgm.dump(os.path.join(os.getcwd(), "../data/tests results/osm duplicates test", "dups_tuples_to_delete.pkl"), dups_tuples_to_delete)

### * add manual review deletion

In [ ]:
to_review_ids = tgm.load(os.path.join(os.getcwd(), "../data/tests results/osm duplicates test", "to_review_ids.pkl"))
print(len(to_review_ids))

6


In [ ]:
dups_tuples_to_delete_manual = [('6543282', '6543273', '192756'), ('59190', '59195', '59065')]

In [ ]:
tgm.dump(os.path.join(os.getcwd(), "../data/tests results/osm duplicates test", "dups_tuples_to_delete_manual.pkl"), dups_tuples_to_delete_manual)

In [ ]:
dups_tuples_to_delete = dups_tuples_to_delete + dups_tuples_to_delete_manual
print(len(dups_tuples_to_delete))

191


In [ ]:
tgm.dump(os.path.join(os.getcwd(), "../data/tests results/osm duplicates test", "dups_tuples_to_delete.pkl"), dups_tuples_to_delete)

In [ ]:
print(len(dups_id_tuples))
print(len(dups_tuples_to_delete))
temp = tgm.complement(dups_id_tuples, dups_tuples_to_delete)
print(len(temp))

1805
191
1614


# save

In [19]:
for k,df in df_by_cntr_fixed.items():
    path = os.path.join(os.getcwd(),f'../data/fixed/{k}/{k}_fixed.pkl')
    tgm.dump(path, df)